In [3]:
from google.colab import drive
import os
import numpy as np
from sklearn.manifold import TSNE
from torchvision import models, transforms
from PIL import Image
import torch
import plotly.graph_objects as go
import base64

# Mount Google Drive
drive.mount('/content/drive')

# Path to the main folder containing shape folders
main_folder_path = '/content/drive/MyDrive/sample shapes/shapes'

# Output folder to save t-SNE plots
output_folder_name = "3 dimensions"
output_folder_path = os.path.join('/content/drive/MyDrive/sample shapes/each shape', output_folder_name)

# Create the output folder if it doesn't exist
os.makedirs(output_folder_path, exist_ok=True)

def load_and_process_images_with_resnet(folder_path):
    """
    Load images and extract features using a pre-trained ResNet model.
    """
    resnet = models.resnet18(pretrained=True)
    resnet.fc = torch.nn.Identity()  # Remove the classification layer
    resnet.eval()  # Set to evaluation mode

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        # Normalize with ImageNet mean and std
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    data = []
    labels = []
    image_paths = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".png"):
            file_path = os.path.join(folder_path, file_name)
            image = Image.open(file_path).convert('RGB')
            image_tensor = preprocess(image).unsqueeze(0)  # Add batch dimension
            with torch.no_grad():
                features = resnet(image_tensor).numpy().flatten()
            data.append(features)
            labels.append(file_name.split(".")[0])  # Use filename without extension as label
            image_paths.append(file_path)
    return np.array(data), labels, image_paths

def convert_image_to_base64(image_path):
    """
    Convert an image to a base64 string.
    """
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode()
        return f"data:image/png;base64,{encoded_string}"

def apply_tsne(data):
    """
    Apply t-SNE for dimensionality reduction to 3 dimensions.
    """
    tsne = TSNE(n_components=3, random_state=42)
    reduced_data = tsne.fit_transform(data)
    return reduced_data

def save_interactive_plot(reduced_data, labels, images, output_file_path, shape_name):
    """
    Save an interactive 3D plot with t-SNE results using Plotly with base64 encoded images.
    """
    # Convert all image paths to base64 strings
    images_base64 = [convert_image_to_base64(image) for image in images]

    # Create a Plotly scatter 3D plot
    fig = go.Figure(data=[go.Scatter3d(
        x=reduced_data[:, 0],
        y=reduced_data[:, 1],
        z=reduced_data[:, 2],
        mode='markers',
        marker=dict(size=5, opacity=0.8),
        text=[f"Label: {label}, Shape number: {index + 1}" for index, label in enumerate(labels)],
        hoverinfo='text'
    )])

    fig.update_layout(
        title=f't-SNE Visualization for {shape_name}',
        scene=dict(
            xaxis_title='t-SNE Dimension 1',
            yaxis_title='t-SNE Dimension 2',
            zaxis_title='t-SNE Dimension 3'
        )
    )

    # Save the plot to an HTML file
    fig.write_html(output_file_path)
    print(f"Saved interactive t-SNE plot to {output_file_path}")

def process_folders_for_tsne(main_folder_path, output_folder_path):
    """
    Process each shape folder, apply t-SNE, and save the plots.
    """
    for shape_folder in os.listdir(main_folder_path):
        shape_folder_path = os.path.join(main_folder_path, shape_folder)

        # Check if it's a valid shape folder
        if os.path.isdir(shape_folder_path) and shape_folder.startswith("shape"):
            print(f"Processing {shape_folder}...")

            data, labels, image_paths = load_and_process_images_with_resnet(shape_folder_path)

            if data.shape[0] == 0:
                print(f"No images found in {shape_folder}. Skipping.")
                continue

            print(f"Loaded and processed {data.shape[0]} images for {shape_folder}.")

            reduced_data = apply_tsne(data)

            # Save the interactive plot
            output_file_name = f"{shape_folder}_tsne_plot.html"
            output_file_path = os.path.join(output_folder_path, output_file_name)
            save_interactive_plot(reduced_data, labels, image_paths, output_file_path, shape_folder)

# Main function
def main():
    # Process each shape folder and generate t-SNE plots
    process_folders_for_tsne(main_folder_path, output_folder_path)

# Run the main function
main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing shape1...
Loaded and processed 213 images for shape1.
Saved interactive t-SNE plot to /content/drive/MyDrive/sample shapes/each shape/3 dimensions/shape1_tsne_plot.html
Processing shape2...


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning:

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning:

Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.



Loaded and processed 214 images for shape2.
Saved interactive t-SNE plot to /content/drive/MyDrive/sample shapes/each shape/3 dimensions/shape2_tsne_plot.html
Processing shape3...


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning:

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning:

Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.



Loaded and processed 214 images for shape3.
Saved interactive t-SNE plot to /content/drive/MyDrive/sample shapes/each shape/3 dimensions/shape3_tsne_plot.html
Processing shape4...


...

Loaded and processed 209 images for shape19.
Saved interactive t-SNE plot to /content/drive/MyDrive/sample shapes/each shape/3 dimensions/shape19_tsne_plot.html
Processing shape20...


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning:

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning:

Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.



Loaded and processed 207 images for shape20.
Saved interactive t-SNE plot to /content/drive/MyDrive/sample shapes/each shape/3 dimensions/shape20_tsne_plot.html
Processing shape21...


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning:

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning:

Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.



Loaded and processed 204 images for shape21.
Saved interactive t-SNE plot to /content/drive/MyDrive/sample shapes/each shape/3 dimensions/shape21_tsne_plot.html
